## dotor-record处理

In [8]:
import pandas as pd
import math

path = "/Users/Desktop/春雨数据整理/数据/"  #需修改

file1 = path + "doctor_records_2022_05_02.xls"
file2 = path + "doctor_records_2022_07_04.xlsx"

# Excel 最大行数限制（.xlsx）
MAX_ROWS = 1048575

sheet1 = "送心意明细"
sheet2 = "热门咨询明细"
sheet3 = "热门咨询明细-分页1"


# ========= 读取 =========
f1_s1 = pd.read_excel(file1, sheet_name=sheet1)
f1_s2 = pd.read_excel(file1, sheet_name=sheet2)
f1_s3 = pd.read_excel(file1, sheet_name=sheet3)

f2_s1 = pd.read_excel(file2, sheet_name=sheet1)
f2_s2 = pd.read_excel(file2, sheet_name=sheet2)
f2_s3 = pd.read_excel(file2, sheet_name=sheet3)

In [9]:
# ========= Sheet1：直接并集去重 =========
final_s1 = (
    pd.concat([f1_s1, f2_s1], ignore_index=True)
    .drop_duplicates()
)


# ========= Sheet2+3：各自先合并，再总体合并 =========
data1 = pd.concat([f1_s2, f1_s3], ignore_index=True)
data2 = pd.concat([f2_s2, f2_s3], ignore_index=True)

final_big = (
    pd.concat([data1, data2], ignore_index=True)
    .drop_duplicates()
)

In [10]:
# ========= 自动分页函数 =========
def split_dataframe(df, base_name):
    sheets = {}
    n_parts = math.ceil(len(df) / MAX_ROWS)

    for i in range(n_parts):
        start = i * MAX_ROWS
        end = (i + 1) * MAX_ROWS
        name = base_name if i == 0 else f"{base_name}-分页{i}"
        sheets[name] = df.iloc[start:end]

    return sheets


big_sheets = split_dataframe(final_big, sheet2)

In [11]:
# ========= 导出 =========
output_file = path + "doctor_records_merged.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final_s1.to_excel(writer, sheet_name=sheet1, index=False)

    for name, df in big_sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)


print("✅ 合并完成，输出文件：", output_file)
print("Sheet1行数：", len(final_s1))
print("大表总行数：", len(final_big))

✅ 合并完成，输出文件： /Users/bytedance/Desktop/春雨数据整理/数据/doctor_records_merged.xlsx
Sheet1行数： 385373
大表总行数： 1449040


## 数据清洗

In [14]:
import glob

file = path + "doctor_records_merged.xlsx"

# ========= 读取（自动合并分页sheet） =========
xls = pd.ExcelFile(file)

# 只读取 热门咨询明细 相关sheet
target_sheets = [s for s in xls.sheet_names if s.startswith("热门咨询明细")]

df_list = [pd.read_excel(file, sheet_name=s) for s in target_sheets]
df = pd.concat(df_list, ignore_index=True)

print("原始行数：", len(df))

原始行数： 1449040


In [15]:
# ========= 1. 时间过滤 =========以创建时间为准
df["创建时间"] = pd.to_datetime(df["创建时间"], errors="coerce")

df = df[
    (df["创建时间"] >= "2022-01-03") &
    (df["创建时间"] <= "2022-06-27")
]

print("时间过滤后：", len(df))

时间过滤后： 160441


In [5]:
"""
# ========= 2. 删除隐私失败记录 =========
df = df[df["对话"] != "[患者隐私，不予显示]"]

print("删除失败记录后：", len(df))
"""

语音筛选后： 35346


In [ ]:
"""
# ========= 3. 只保留有语音 =========
voice_pattern = r"\[语音\(.*?\.mp3\)\]"
df = df[df["对话"].str.contains(voice_pattern, regex=True, na=False)]

print("语音筛选后：", len(df))
"""

In [16]:
# 按照id+创建时间进行排序
df = df.sort_values(
    by=['id', '创建时间'],    # 先id，后创建时间
    ascending=[True, True]   # 都升序（从小到大/从早到晚）
)
df = df.reset_index(drop=True)

In [17]:
# ========= 4. 导出 =========
output_file = path + "doctor_records_cleaned.xlsx"
df.to_excel(output_file, index=False)

print("✅ 清洗完成，输出：", output_file)

✅ 清洗完成，输出： /Users/bytedance/Desktop/春雨数据整理/数据/doctor_records_cleaned.xlsx


In [27]:
voice_pattern = r"\[语音\(.*?\.mp3\)\]"
filtered = df[df["对话"].str.contains(voice_pattern, regex=True, na=False)]

print("语音筛选后数量：", len(filtered))

# 打印一条样例（不改动原数据）
if len(filtered) > 0:
    print("\n一条语音对话样例：")
    print(filtered["对话"].iloc[0])
else:
    print("\n未找到语音内容")

语音筛选后数量： 17185

一条语音对话样例：
[2022-01-03 06:17:17][患者]: 甲钩炎肿了汇脓了怎么办（2，15）
[2022-01-03 06:17:17][患者]: [1张图片]
[2022-01-03 06:17:27][医生]: 您好，我是胡医生，您的问题已经收到，感谢您对我的信任！如果您方便，可再详细描诉一下您咨询的相关病史、近期的身体主要不适症状、外院诊疗经过、既往病史情况等等，我好仔细评估及答复您。谢谢理解。
[2022-01-03 06:17:39][医生]: 您好，你的情况需要尽快切开排脓
[2022-01-03 06:17:47][医生]: 每日换药
[2022-01-03 06:18:12][医生]: 而且要修剪掉部分趾甲
[2022-01-03 18:56:41][患者]: 大夫您好还在吗
[2022-01-03 18:58:19][患者]: 今天带孩子去了医院只给我开了药膏什么也没说，让我回家自己处理
[2022-01-03 18:58:57][患者]: 您看这样能行吗
[2022-01-03 18:58:57][患者]: [1张图片]
[2022-01-03 19:01:09][患者]: 一周前剪指甲剪的有点短了，天天穿着棉鞋上学，就有点感染了，昨天开始就看到有脓了
[2022-01-03 19:30:07][医生]: [语音(https://resourced.chunyu.mobi/37UAAAB0cYtCv8YW-b3583091-44e2-4e4d-9859-2660fd16d15f.mp3)]
[2022-01-03 20:04:38][患者]: 好的我试试
[2022-01-03 20:06:40][医生]: 要让医生弄放心一点
[2022-01-03 20:06:50][医生]: 自己弄可能没有那么好的条件哦
[2022-01-03 20:11:29][患者]: 今天的大夫根本没管我们
[2022-01-03 20:11:36][患者]: 感觉更严重了
[2022-01-03 20:13:21][医生]: 可以去急诊外科处理一下
[2022-01-03 20:13:33][医生]: 急诊外科清创挑开排脓一下就可以
[2022-01-03 20:13:45][患者]: 能看到图片吗
[2022-01-

In [30]:
import re
import pandas as pd

# 定义正则：匹配第一条 [医生] 的时间
def get_first_doctor_time(text):
    # 匹配格式：[2022-01-03 06:17:27][医生]:
    pattern = r'\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\[医生\]'
    match = re.search(pattern, str(text))
    if match:
        return pd.to_datetime(match.group(1))  # 转成时间类型
    return pd.NaT

# ========== 核心：不修改原 df，只做计算 ==========
# 1. 计算每条对话的第一条医生回复时间
df_temp = df.copy()
df_temp['第一条医生时间'] = df_temp['对话'].apply(get_first_doctor_time)

# 2. 确保创建时间也是 datetime 类型
df_temp['创建时间'] = pd.to_datetime(df_temp['创建时间'], errors='coerce')

# 3. 筛选：医生第一条回复时间 **早于** 创建时间 的行
result = df_temp[
    (df_temp['第一条医生时间'].notna()) & 
    (df_temp['第一条医生时间'] < df_temp['创建时间'])
]

# ========== 输出结果 ==========
print(f"符合条件的行数：{len(result)}")

if len(result) > 0:
    print("\n第一条符合条件的样例：")
    print("创建时间：", result['创建时间'].iloc[0])
    print("第一条医生回复时间：", result['第一条医生时间'].iloc[0])

符合条件的行数：4613

第一条符合条件的样例：
创建时间： 2022-02-18 15:30:03
第一条医生回复时间： 2022-02-16 16:51:36


## 计算文本衍生变量

In [29]:
"""
医生问诊对话特征提取与按(id, 周)聚合脚本
"""

from typing import List, Dict
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta


# ============================================================
# 1. 对话解析函数
# ============================================================
def parse_dialogue(dialogue_text: str) -> List[Dict]:
    """解析对话文本，提取每条消息的结构化信息"""
    if pd.isna(dialogue_text) or dialogue_text == '':
        return []
    messages = []
    
    pattern = r'\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\[(患者|医生)\]:\s*(.*?)(?=\n\s*\[|$)'
    matches = re.finditer(pattern, dialogue_text, re.DOTALL)
    
    for match in matches:
        timestamp_str = match.group(1)
        sender = match.group(2)
        content = match.group(3).strip()
        
        try:
            timestamp = datetime.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')
        except:
            timestamp = None
        
        # 图片检测
        has_image = bool(re.search(r'\[(\d+)张图片\]', content))
        image_count = 0
        if has_image:
            img_match = re.search(r'\[(\d+)张图片\]', content)
            if img_match:
                image_count = int(img_match.group(1))
        
        # 语音检测 —— 兼容中文括号（）和英文括号()
        has_voice = bool(re.search(r'\[语音[（(].*?[)）]\]', content))
        
        # 提取纯文本（去除图片、语音标记）
        text_content = re.sub(r'\[\d+张图片\]', '', content)
        text_content = re.sub(r'\[语音[（(].*?[)）]\]', '', text_content)
        text_content = text_content.strip()
        
        # 统计表情符号
        emoji_count = 0
        emoji_count += len(re.findall(r'[\U0001F600-\U0001F64F]', content))
        emoji_count += len(re.findall(r'[\U0001F300-\U0001F5FF]', content))
        emoji_count += len(re.findall(r'[\U0001F680-\U0001F6FF]', content))
        emoji_count += len(re.findall(r'[\U0001F900-\U0001F9FF]', content))
        
        messages.append({
            'timestamp': timestamp,
            'sender': sender,
            'content': content,
            'text_content': text_content,
            'text_length': len(text_content),
            'has_image': has_image,
            'image_count': image_count,
            'has_voice': has_voice,
            'emoji_count': emoji_count
        })
    
    return messages


# ============================================================
# 2. 医生特征提取函数
# ============================================================
def extract_doctor_features_from_dialogue(dialogue_text: str, creation_time=None) -> Dict:
    """从单条对话中提取医生相关特征
    
    修改说明:
    1. 医生总回复时长: 只在患者->医生的首次切换时计算一次回复时长，
       医生连续发送多条消息不重复计算。单位为秒。
    2. 新增: 医生首次回复距创建时间的差（秒）。
    3. 医生文本条数 / 医生文本平均字数: 仅统计有实际文本内容的消息
       （排除纯图片、纯语音等无文字内容的消息）。
    """
    messages = parse_dialogue(dialogue_text)
    
    if not messages:
        return {
            '医生文本条数': 0,
            '医生总字数': 0,
            '医生文本平均字数': 0,
            '医生回复次数': 0,
            '医生总回复时长': 0,
            '医生平均回复时长': 0,
            '医生首次回复距创建时间_秒': np.nan,
            '对话图片数': 0,
            '对话表情数': 0,
            '医生语音条数': 0,
            '医生最大连续语音数': 0,
        }
    
    doctor_messages = [m for m in messages if m['sender'] == '医生']
    
    # --- 修改点3: 仅统计有实际文本内容的医生消息 ---
    # 有实际文本内容 = text_length > 0（去除图片/语音标记后仍有文字）
    doctor_text_messages = [m for m in doctor_messages if m['text_length'] > 0]
    doctor_text_count = len(doctor_text_messages)
    doctor_total_chars = sum(m['text_length'] for m in doctor_text_messages)
    doctor_avg_chars = doctor_total_chars / doctor_text_count if doctor_text_count > 0 else 0
    
    # --- 修改点1: 医生回复时长，避免连续回复重复计算 ---
    # 逻辑: 遍历消息序列，当发生"患者 -> 医生"的角色切换时，
    #        用医生该条消息的时间 - 前面最近一条患者消息的时间 = 一次回复时长。
    #        医生连续发送的后续消息不再重复计算。
    reply_times = []
    last_patient_timestamp = None  # 记录最近一条患者消息的时间
    last_sender = None  # 记录上一条消息的发送者
    
    for msg in messages:
        if msg['sender'] == '患者' and msg['timestamp']:
            last_patient_timestamp = msg['timestamp']
        elif msg['sender'] == '医生' and msg['timestamp']:
            # 只在 "患者 -> 医生" 切换时计算（即上一条是患者）
            if last_sender == '患者' and last_patient_timestamp is not None:
                time_diff = (msg['timestamp'] - last_patient_timestamp).total_seconds()
                if time_diff >= 0:
                    reply_times.append(time_diff)
        last_sender = msg['sender']
    
    doctor_reply_count = len(reply_times)
    doctor_total_reply_time = sum(reply_times)  # 单位: 秒
    doctor_avg_reply_time = np.mean(reply_times) if reply_times else 0
    
    # --- 修改点2: 医生首次回复距创建时间的差 ---
    doctor_first_reply_from_creation = np.nan
    if creation_time is not None and not pd.isna(creation_time):
        # 找到医生的第一条消息
        for msg in messages:
            if msg['sender'] == '医生' and msg['timestamp']:
                time_diff = (msg['timestamp'] - creation_time).total_seconds()
                doctor_first_reply_from_creation = time_diff
                break
    
    total_images = sum(m['image_count'] for m in messages)
    total_emojis = sum(m['emoji_count'] for m in messages)
    doctor_voice_count = sum(1 for m in doctor_messages if m['has_voice'])
    
    # 最大连续语音数
    max_consecutive_voice = 0
    current_consecutive = 0
    for msg in messages:
        if msg['sender'] == '医生' and msg['has_voice']:
            current_consecutive += 1
            max_consecutive_voice = max(max_consecutive_voice, current_consecutive)
        else:
            current_consecutive = 0
    
    return {
        '医生文本条数': doctor_text_count,
        '医生总字数': doctor_total_chars,
        '医生文本平均字数': doctor_avg_chars,
        '医生回复次数': doctor_reply_count,
        '医生总回复时长': doctor_total_reply_time,
        '医生平均回复时长': doctor_avg_reply_time,
        '医生首次回复距创建时间_秒': doctor_first_reply_from_creation,
        '对话图片数': total_images,
        '对话表情数': total_emojis,
        '医生语音条数': doctor_voice_count,
        '医生最大连续语音数': max_consecutive_voice,
    }


# ============================================================
# 3. 周划分函数
# ============================================================
def assign_week(creation_time: pd.Timestamp) -> pd.Timestamp:
    """根据创建时间分配所属周的周一日期"""
    if pd.isna(creation_time):
        return pd.NaT
    days_since_monday = creation_time.weekday()
    week_monday = creation_time - timedelta(days=days_since_monday)
    return pd.Timestamp(week_monday.date())

# ---------- 读取数据 ----------
file = path + "doctor_records_cleaned.xlsx"

print("=" * 60)
print("📂 读取数据...")
df = pd.read_excel(file)
print(f"  原始数据: {df.shape[0]} 行, {df.shape[1]} 列")
print(f"  字段: {list(df.columns)}")

# ---------- 划分周 ----------
print("\n📅 划分问诊所属周...")
df['周'] = df['创建时间'].apply(assign_week)
print(f"  周数范围: {df['周'].min()} ~ {df['周'].max()}")
print(f"  共 {df['周'].nunique()} 个周")

# ---------- 语音格式预检 ----------
print("\n🔎 语音格式预检...")
sample_with_voice = df[df['对话'].astype(str).str.contains('语音', na=False)].head(5)
print(f"  含'语音'关键字的记录数: {df['对话'].astype(str).str.contains('语音', na=False).sum()}")

# 检查括号类型
cn_paren_count = df['对话'].astype(str).str.contains(r'\[语音（', na=False).sum()
en_paren_count = df['对话'].astype(str).str.contains(r'\[语音\(', na=False).sum()
print(f"  中文括号（）格式: {cn_paren_count} 条")
print(f"  英文括号()格式: {en_paren_count} 条")

# ---------- 提取对话特征 ----------
print(f"\n🔍 提取对话特征 (共 {len(df)} 条记录)...")

features_list = []
total = len(df)
for i, (idx, row) in enumerate(df.iterrows()):
    if (i + 1) % 10000 == 0 or i == 0:
        print(f"  进度: {i+1}/{total} ({(i+1)/total*100:.1f}%)")
    features = extract_doctor_features_from_dialogue(
        row.get('对话', ''),
        creation_time=row.get('创建时间', None)
    )
    features_list.append(features)

features_df = pd.DataFrame(features_list, index=df.index)
df = pd.concat([df, features_df], axis=1)
print(f"  ✅ 特征提取完成，新增 {len(features_df.columns)} 个特征列")

# 语音特征验证
voice_records = df[df['医生语音条数'] > 0]
print(f"  📢 含医生语音的问诊记录: {len(voice_records)} 条")
if len(voice_records) > 0:
    print(f"     医生语音条数分布: mean={voice_records['医生语音条数'].mean():.2f}, "
          f"max={voice_records['医生语音条数'].max()}")

# 首次回复特征验证
valid_first_reply = df[df['医生首次回复距创建时间_秒'].notna()]
print(f"  ⏱️ 有医生首次回复时间的记录: {len(valid_first_reply)} 条")
if len(valid_first_reply) > 0:
    print(f"     首次回复时间(秒)分布: mean={valid_first_reply['医生首次回复距创建时间_秒'].mean():.2f}, "
          f"median={valid_first_reply['医生首次回复距创建时间_秒'].median():.2f}, "
          f"max={valid_first_reply['医生首次回复距创建时间_秒'].max():.2f}")

# ---------- 按 (id, 周) 聚合 ----------
print(f"\n📊 按 (id, 周) 聚合...")

agg_dict = {
    # 基本信息
    '姓名': 'first',
    
    # 对话特征 - 求均值
    '医生文本条数': 'mean',
    '医生文本平均字数': 'mean',
    '医生平均回复时长': 'mean',
    '医生首次回复距创建时间_秒': 'mean',
    '对话图片数': 'mean',
    '对话表情数': 'mean',
    '医生语音条数': 'mean',
    
    # 最大连续语音 - 取 max
    '医生最大连续语音数': 'max',
    
    # 本周问诊次数
    '对话': 'count',
}

# 只保留 df 中实际存在的列
agg_dict_filtered = {k: v for k, v in agg_dict.items() if k in df.columns}

panel_weekly = df.groupby(['id', '周']).agg(agg_dict_filtered).reset_index()

# 重命名
panel_weekly.rename(columns={
    '对话': '本周问诊次数',
    '医生文本条数': '医生发送文本平均条数',
    '医生文本平均字数': '医生文本平均长度_字数',
    '医生平均回复时长': '医生回复平均速度_秒',
    '医生首次回复距创建时间_秒': '医生首次回复距创建时间平均_秒',
    '对话图片数': '对话中含图片平均数量',
    '对话表情数': '表情符号平均数量',
    '医生语音条数': '医生发送语音平均条数',
    '医生最大连续语音数': '最多连续发送语音数',
}, inplace=True)

# ---------- 打印统计 ----------
print(f"  ✅ 聚合完成: {panel_weekly.shape[0]} 行 (医生-周 组合)")
print(f"  唯一医生数: {panel_weekly['id'].nunique()}")
print(f"  唯一周数: {panel_weekly['周'].nunique()}")

print(f"\n📈 聚合后特征统计:")
feature_cols = [
    '本周问诊次数',
    '医生发送文本平均条数',
    '医生文本平均长度_字数',
    '医生回复平均速度_秒',
    '医生首次回复距创建时间平均_秒',
    '对话中含图片平均数量',
    '表情符号平均数量',
    '医生发送语音平均条数',
    '最多连续发送语音数',
]

for col in feature_cols:
    if col in panel_weekly.columns:
        mean_val = panel_weekly[col].mean()
        median_val = panel_weekly[col].median()
        std_val = panel_weekly[col].std()
        print(f"  {col:35s}: 均值={mean_val:8.2f}, 中位数={median_val:8.2f}, 标准差={std_val:8.2f}")

📂 读取数据...
  原始数据: 160441 行, 8 列
  字段: ['id', '姓名', '用户id', '创建时间', '评价时间', '评价', '标签', '对话']

📅 划分问诊所属周...
  周数范围: 2022-01-03 00:00:00 ~ 2022-06-20 00:00:00
  共 25 个周

🔎 语音格式预检...
  含'语音'关键字的记录数: 19229
  中文括号（）格式: 0 条
  英文括号()格式: 17185 条

🔍 提取对话特征 (共 160441 条记录)...
  进度: 1/160441 (0.0%)
  进度: 10000/160441 (6.2%)
  进度: 20000/160441 (12.5%)
  进度: 30000/160441 (18.7%)
  进度: 40000/160441 (24.9%)
  进度: 50000/160441 (31.2%)
  进度: 60000/160441 (37.4%)
  进度: 70000/160441 (43.6%)
  进度: 80000/160441 (49.9%)
  进度: 90000/160441 (56.1%)
  进度: 100000/160441 (62.3%)
  进度: 110000/160441 (68.6%)
  进度: 120000/160441 (74.8%)
  进度: 130000/160441 (81.0%)
  进度: 140000/160441 (87.3%)
  进度: 150000/160441 (93.5%)
  进度: 160000/160441 (99.7%)
  ✅ 特征提取完成，新增 11 个特征列
  📢 含医生语音的问诊记录: 17185 条
     医生语音条数分布: mean=6.13, max=76
  ⏱️ 有医生首次回复时间的记录: 135790 条
     首次回复时间(秒)分布: mean=-7386.12, median=123.00, max=156269.00

📊 按 (id, 周) 聚合...
  ✅ 聚合完成: 43473 行 (医生-周 组合)
  唯一医生数: 4740
  唯一周数: 25

📈 聚合后特征统计:
  本周问诊次数             

In [20]:
# ---------- 保存 ----------
output_file = path + "panel_weekly_features.xlsx"
panel_weekly.to_excel(output_file, index=False)
print(f"\n💾 结果已保存至: {output_file}")

detail_file = path + "doctor_records_with_features.xlsx"
df.to_excel(detail_file, index=False)
print(f"💾 明细数据已保存至: {detail_file}")

print("\n" + "=" * 60)
print("✅ 全部完成！")


💾 结果已保存至: /Users/bytedance/Desktop/春雨数据整理/数据/panel_weekly_features.xlsx
💾 明细数据已保存至: /Users/bytedance/Desktop/春雨数据整理/数据/doctor_records_with_features.xlsx

✅ 全部完成！
